# 异质性分析-市值  
将AED因子数据分割为大市值组和小市值组，
分别进行组合构建，分桶，FM回归  

## 导入库

In [1]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
dotenv.load_dotenv()

True

## 超参数

In [ ]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'baseline1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

# 异质性组
MV_GROUP = 0 # 0 高市值组； 1 低市值


## 读取数据
读取MA因子数据和市值数据，划分为大市值组和小市值组

In [3]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552


In [4]:
market_value_df = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='''SELECT stkcd AS "Stkcd", trdmnt AS "Trdmnt", msmvosd AS "Msmvosd" FROM statics.market_value''',
    engine=ENGINE,
)

## 处理数据

### 市值数据重命名

In [5]:
market_value_df = market_value_df.rename(
    {
        'Stkcd':'portfolio',
        'Trdmnt':'date',
        'Msmvosd':'msmvosd'
    }
)

market_value_df.head()

portfolio,date,msmvosd
str,date,f64
"""000557""",1997-05-01,963245.03
"""000031""",1997-07-01,1.3565e6
"""600835""",1997-09-01,177336.0
"""900920""",1997-03-01,107198.0
"""000417""",1997-06-01,415800.0


## 划分数据
按照date，生成所有portfolio的中位数，然后根据中位数，划分大市值组和小市值组

In [6]:
market_value_df = market_value_df.with_columns(
    pl.col('msmvosd').quantile(1/2).over('date').alias('msmvosd_1')
)
market_value_df = market_value_df.with_columns(
    pl.when(pl.col('msmvosd') > pl.col('msmvosd_1'))
    .then(0)
    .otherwise(1)
    .alias('value_id')
)
market_value_df = market_value_df.drop(['msmvosd','msmvosd_1'])

In [7]:
# 筛选
market_value_df = market_value_df.filter(pl.col('value_id') == MV_GROUP)
market_value_df.head()

portfolio,date,value_id
str,date,i32
"""600835""",1997-09-01,1
"""900920""",1997-03-01,1
"""600710""",1997-06-01,1
"""600075""",1997-07-01,1
"""900936""",1997-01-01,1


和ma_df合并，保存为 MA因子.parquet 文件

In [8]:
ma_df = ma_df.join(market_value_df,on=['portfolio','date'],how='left').drop_nulls('value_id').drop('value_id')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2024-03-01,"""300512""",0.839616,-0.0327
2021-07-01,"""300092""",0.5874,0.234804
2023-12-01,"""688292""",0.728695,-0.2864
2022-12-01,"""002213""",0.5731,0.039


## 保存数据

In [9]:
ma_df.write_parquet(SAVE_BASE_DIR + '/MA因子.parquet')

**保存后，运行2，3，4 notebook，查看异质性结果** 